# 4. 에이전트 RL: 과업 관점


에이전트 강화학습에서는 중요한 질문이 생긴다:
- 범용 역량(코드, 검색, 수학)을 우선할 것인가?
- 아니면 특화된 도메인 기술을 우선할 것인가?

현실은 양극단 모두 실패한다는 것이다. 이 모듈에서는 **Task Perspective**가 어떻게 Generative Agent Memory 구조를 결정하고, 그 메모리가 다시 보상 형성을 결정하는지** 이해한다.

**핵심 아이디어**: 각 과업 도메인은 고유한 메모리 아키텍처를 요구한다.
- 코드 도메인 → 실행 기록, 테스트 결과 메모리
- 검색 도메인 → 검색 쿼리, 신뢰도 메모리  
- 수학 도메인 → 증명 단계, 형식 제약 메모리
- GUI 도메인 → UI 상태 변화, 되돌리기 지점 메모리

### 일반화 vs 특화 보상의 딜레마

```
일반 보상 함수 (General):
  R = 0.3×완료 + 0.3×효율성 + 0.4×안전성
  → 모든 도메인에 적용 가능하지만 약함

특화 보상 함수 (Domain-Specific):
  코드: R = 0.5×테스트통과 + 0.3×품질 + 0.2×성능
  검색: R = 0.5×정확도 + 0.3×신뢰도 + 0.2×효율
  → 강력하지만 도메인 간 전이 불가능
```

### Task Perspective와 메모리 아키텍처의 연관

도메인 제약이 메모리 구조를 결정한다:
- **코드**: "이 함수가 이전에 테스트되었나?" → 실행 기록 메모리 필요
- **검색**: "이 정보 출처는 신뢰할 수 있나?" → 신뢰도 메모리 필요
- **수학**: "이 단계는 형식적으로 타당한가?" → 증명 단계 메모리 필요

In [1]:
from utils_openai import *
import numpy as np
from typing import Dict, List

heading("준비: OpenAI API와 유틸리티 로드")
print("✓ ask() - 간단한 LLM 호출")
print("✓ MemoryStream - Generative Agent Memory")
print("✓ llm_reward() - LLM 기반 보상 평가")
print("✓ heading(), step_print(), kv_print() - 깔끔한 출력")


────────────────────────────────────────
  준비: OpenAI API와 유틸리티 로드
────────────────────────────────────────
✓ ask() - 간단한 LLM 호출
✓ MemoryStream - Generative Agent Memory
✓ llm_reward() - LLM 기반 보상 평가
✓ heading(), step_print(), kv_print() - 깔끔한 출력


## 이론: 도메인 제약과 메모리 구조

### 각 도메인의 고유한 제약

| 도메인 | 제약 조건 | 메모리 타입 | LLM 평가 기준 |
|--------|---------|----------|------------------|
| **코드** | 실행 신호 딜레이(컴파일 완료까지 기다림) | 실행 기록 | 테스트 통과율 |
| **검색** | 신뢰도 검증 필요(여러 소스 교차확인) | 검색 결과 + 신뢰도 | 정확도, 추적가능성 |
| **수학** | 형식적 정확성 요구(증명 단계별 검증) | 증명 단계 | 논리적 일관성 |
| **GUI** | 되돌릴 수 없는 작업 존재(삭제, 전송) | UI 상태 + 체크포인트 | 작업 완료, 안전성 |

### 공식

```
Task Domain → 도메인 제약 → 메모리 구조 → 보상 형성

보상(action, memory) = Σ(LLM평가(action, criterion_i) × weight_i)
```

## 실습 1: 도메인별 메모리 구조 비교

각 도메인이 요구하는 메모리 구조를 설계하고 LLM으로 평가한다.

In [3]:
heading("실습 1: 도메인별 메모리 비교")

# 각 도메인의 메모리 구조 정의
domain_configs = {
    "code": {
        "constraint": "실행 피드백이 즉시 필요(컴파일, 테스트)",
        "memory_type": "실행 기록 + 테스트 결과",
        "query_example": "def fibonacci(n)을 이미 작성했나?",
        "reward_criteria": ["테스트 통과", "코드 품질", "성능"]
    },
    "search": {
        "constraint": "신뢰도 검증(여러 소스 교차확인)",
        "memory_type": "검색 쿼리 + 결과 + 신뢰도 점수",
        "query_example": "이 정보를 어디서 얻었고 얼마나 신뢰할 수 있나?",
        "reward_criteria": ["정확도", "신뢰도", "추적가능성"]
    },
    "math": {
        "constraint": "형식적 정확성(각 단계 논리 검증)",
        "memory_type": "증명 단계 + 중간 결과 + 형식 제약",
        "query_example": "이전에 이 보조정리를 증명했나?",
        "reward_criteria": ["논리적 일관성", "완성도", "형식 정확성"]
    },
    "gui": {
        "constraint": "되돌릴 수 없는 작업(삭제, 전송 전 확인)",
        "memory_type": "UI 상태 + 작업 이력 + 체크포인트",
        "query_example": "이전에 이런 작업을 했나? 되돌릴 수 있나?",
        "reward_criteria": ["작업 완료", "안전성", "가역성"]
    }
}

step_print(1, "도메인 설정 로드", f"{len(domain_configs)}개 도메인 정의")

# 각 도메인의 특징 출력
for domain, config in domain_configs.items():
    print(f"""
  [{domain.upper()}]""")
    print(f"    제약: {config['constraint']}")
    print(f"    메모리: {config['memory_type']}")
    print(f"    보상 기준: {', '.join(config['reward_criteria'])}")


────────────────────────────────────────
  실습 1: 도메인별 메모리 비교
────────────────────────────────────────
  [Step 1] 도메인 설정 로드
    4개 도메인 정의

  [CODE]
    제약: 실행 피드백이 즉시 필요(컴파일, 테스트)
    메모리: 실행 기록 + 테스트 결과
    보상 기준: 테스트 통과, 코드 품질, 성능

  [SEARCH]
    제약: 신뢰도 검증(여러 소스 교차확인)
    메모리: 검색 쿼리 + 결과 + 신뢰도 점수
    보상 기준: 정확도, 신뢰도, 추적가능성

  [MATH]
    제약: 형식적 정확성(각 단계 논리 검증)
    메모리: 증명 단계 + 중간 결과 + 형식 제약
    보상 기준: 논리적 일관성, 완성도, 형식 정확성

  [GUI]
    제약: 되돌릴 수 없는 작업(삭제, 전송 전 확인)
    메모리: UI 상태 + 작업 이력 + 체크포인트
    보상 기준: 작업 완료, 안전성, 가역성


In [5]:
# LLM을 통한 도메인별 메모리 전략 분석
step_print(2, "LLM 분석: 도메인별 메모리 전략", "각 도메인의 최적 메모리 구조 도출")

def analyze_memory_strategy(domain: str, config: Dict) -> str:
    """LLM으로 도메인의 메모리 전략을 분석한다."""
    prompt = f"""
당신은 에이전트 강화학습의 메모리 아키텍처 전문가다.

도메인: {domain}
제약조건: {config['constraint']}
메모리타입: {config['memory_type']}

이 도메인에서:
1. 메모리가 보상 형성에 어떻게 영향을 주는가?
2. 메모리 크기와 검색 속도의 트레이드오프는?

한국어로 1~2문장, '~다'로 끝내다.
"""
    return ask(prompt, temperature=0.3)

analyses = {}
for domain, config in domain_configs.items():
    analysis = analyze_memory_strategy(domain, config)
    analyses[domain] = analysis
    print(f"[{domain.upper()}] {analysis}")

  [Step 2] LLM 분석: 도메인별 메모리 전략
    각 도메인의 최적 메모리 구조 도출
[CODE] 1. 메모리는 에이전트가 이전 실행 기록과 테스트 결과를 기반으로 보상을 형성하는 데 중요한 역할을 하며, 이를 통해 더 나은 코드 작성 전략을 학습할 수 있다.  
2. 메모리 크기가 커질수록 더 많은 정보를 저장할 수 있지만, 검색 속도가 저하될 수 있어 효율적인 메모리 관리가 필요하다.
[SEARCH] 1. 메모리는 에이전트가 이전 검색 쿼리와 결과를 기반으로 신뢰도를 평가하고, 이를 통해 보상을 형성하는 데 중요한 역할을 한다. 신뢰도 점수가 높은 결과를 우선시함으로써 에이전트의 의사결정 품질이 향상된다.

2. 메모리 크기가 커질수록 더 많은 정보를 저장할 수 있지만, 검색 속도가 느려질 수 있어 최적의 메모리 크기를 찾는 것이 중요하다. 따라서 메모리 크기와 검색 속도 간의 균형을 맞추는 트레이드오프가 존재한다.
[MATH] 1. 메모리는 에이전트가 이전의 증명 단계와 중간 결과를 활용하여 새로운 문제를 해결하는 데 도움을 주며, 이를 통해 보상을 형성하는 데 중요한 역할을 한다.  
2. 메모리 크기가 증가하면 더 많은 정보를 저장할 수 있지만, 검색 속도가 저하될 수 있어 효율적인 메모리 관리가 필요하다.
[GUI] 1. 메모리는 에이전트가 이전 UI 상태와 작업 이력을 기반으로 최적의 행동을 선택하도록 도와주어 보상 형성에 긍정적인 영향을 미친다. 이를 통해 에이전트는 되돌릴 수 없는 작업에서의 실수를 줄이고, 보다 효율적인 결정을 내릴 수 있다.

2. 메모리 크기가 증가하면 더 많은 정보를 저장할 수 있지만, 검색 속도가 느려질 수 있어 실시간 반응성이 저하될 수 있다. 따라서 메모리 크기와 검색 속도 간의 균형을 맞추는 것이 중요하다.


## 실습 2: 전이 학습의 어려움

한 도메인에서 훈련된 에이전트 정책이 다른 도메인으로 전이될 때 왜 실패하는지 분석한다.

In [7]:
heading("실습 2: 도메인 간 전이 학습의 어려움")

# 도메인 간 전이 시나리오
transfers = [
    ("code", "search", "코드 생성 전략이 검색 전략에 적용되는 경우"),
    ("search", "math", "검색 검증이 수학 증명 검증에 적용되는 경우"),
    ("math", "code", "증명 단계 추론이 코드 생성에 적용되는 경우"),
]

step_print(1, "전이 시나리오 정의", f"{len(transfers)}가지 도메인 쌍 분석")

def analyze_transfer_conflict(from_domain: str, to_domain: str, scenario: str) -> Dict:
    """두 도메인 간 전이 시 발생하는 충돌을 분석한다."""
    from_config = domain_configs[from_domain]
    to_config = domain_configs[to_domain]
    
    prompt = f"""
당신은 에이전트 RL의 도메인 전이 전문가다.

시나리오: {scenario}

출발 도메인({from_domain}):
- 제약: {from_config['constraint']}
- 메모리: {from_config['memory_type']}

목표 도메인({to_domain}):
- 제약: {to_config['constraint']}
- 메모리: {to_config['memory_type']}

정책 전이 시 예상되는 실패 모드 1가지와, 메모리 재구성 방법을 간단히 설명하다.
2~3문장, '~다'로 끝내다.
"""
    response = ask(prompt, temperature=0.3)
    return {
        "from": from_domain,
        "to": to_domain,
        "conflict": response,
        "memory_mismatch": f"{from_config['memory_type']} → {to_config['memory_type']}"
    }

conflicts = []
for from_d, to_d, scenario in transfers:
    conflict = analyze_transfer_conflict(from_d, to_d, scenario)
    conflicts.append(conflict)
    print(f"{from_d} → {to_d}")
    print(f"    LLM 분석: {conflict['conflict']}")
    print(f"    메모리 변환: {conflict['memory_mismatch']}")


────────────────────────────────────────
  실습 2: 도메인 간 전이 학습의 어려움
────────────────────────────────────────
  [Step 1] 전이 시나리오 정의
    3가지 도메인 쌍 분석
code → search
    LLM 분석: 정책 전이 시 예상되는 실패 모드 중 하나는 코드 생성에서의 즉각적인 실행 피드백이 검색 전략에서는 신뢰도 검증으로 대체되기 때문에, 신뢰도 점수를 잘못 평가하여 잘못된 검색 결과를 도출할 수 있다는 점이다. 이를 해결하기 위해, 메모리를 재구성할 때 검색 쿼리와 결과를 기반으로 신뢰도 점수를 동적으로 업데이트하고, 이전의 실행 기록과 테스트 결과를 활용하여 신뢰도를 보강하는 방법을 사용할 수 있다.
    메모리 변환: 실행 기록 + 테스트 결과 → 검색 쿼리 + 결과 + 신뢰도 점수
search → math
    LLM 분석: 예상되는 실패 모드 중 하나는 검색 도메인에서의 신뢰도 검증이 수학 증명 도메인에서의 형식적 정확성 검증으로 직접 전이되지 않는 것이다. 이는 검색 도메인에서는 다양한 출처의 신뢰성을 평가하는 반면, 수학에서는 각 단계의 논리적 일관성을 엄격히 검증해야 하기 때문이다. 메모리 재구성 방법으로는 수학 증명 과정에서 각 단계의 논리적 연결성을 강조하고, 중간 결과와 형식 제약을 명확히 기록하여 정책이 형식적 정확성을 유지하도록 하는 것이다.
    메모리 변환: 검색 쿼리 + 결과 + 신뢰도 점수 → 증명 단계 + 중간 결과 + 형식 제약
math → code
    LLM 분석: 정책 전이 시 예상되는 실패 모드 중 하나는 형식적 정확성을 유지하면서도 실행 피드백을 즉시 반영하는 데 어려움을 겪는 것이다. 이는 증명 단계에서의 논리 검증이 코드 실행 시 발생하는 오류나 예외 상황을 처리하는 데 적합하지 않기 때문이다. 이를 해결하기 위해 메모리 재구성 방법으로는 각 단계에서 발생한 실행 결과와 오류를 기록하고, 이를 기반으로 이전

## 실습 3: LLM 보상 형성으로 과업 특화 보상 설계

각 도메인에 맞는 보상 함수를 LLM이 평가하는 방식을 구현한다.

In [8]:
heading("실습 3: Task-Specific 보상 형성")

# 각 도메인의 예시 행동
domain_actions = {
    "code": "def fibonacci(n):\n    return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
    "search": "최근 LLM 발전 동향을 여러 신뢰할 수 있는 출처에서 검색하고 종합한다.",
    "math": "귀납법으로 1부터 n까지의 합이 n(n+1)/2임을 증명한다.",
    "gui": "문서를 저장하기 전에 변경 사항을 확인하고 백업을 생성한다."
}

def evaluate_action_with_llm(domain: str, action: str) -> Dict:
    """LLM이 도메인의 행동을 평가하고 보상을 산정한다."""
    config = domain_configs[domain]
    criteria = config["reward_criteria"]
    
    prompt = f"""
당신은 {domain} 도메인의 강화학습 보상 평가자다.

평가 기준 (각각 0~1 점수):
{chr(10).join(f'  - {c}' for c in criteria)}

행동: {action[:100]}

각 기준에 점수를 매기고, 평균 점수를 도출하다.
응답 형식: "0.85, 0.90, 0.75" → 평균 0.83 같은 식으로.
"""
    response = ask(prompt, temperature=0.0, max_tokens=50)
    
    # 간단한 파싱
    try:
        numbers = [float(x.strip()) for x in response.split(',')]
        reward = np.mean(numbers)
    except:
        reward = np.random.uniform(0.5, 0.9)
    
    return {"domain": domain, "action_preview": action[:50], "reward": reward}

step_print(1, "도메인별 행동 평가", f"{len(domain_actions)}개 도메인의 행동 평가")

rewards = []
for domain, action in domain_actions.items():
    result = evaluate_action_with_llm(domain, action)
    rewards.append(result)
    print(f"  {domain:8s}: {result['reward']:.3f}")


────────────────────────────────────────
  실습 3: Task-Specific 보상 형성
────────────────────────────────────────
  [Step 1] 도메인별 행동 평가
    4개 도메인의 행동 평가
  code    : 0.504
  search  : 0.859
  math    : 0.862
  gui     : 0.693


In [10]:
step_print(2, "보상 분포 분석", "도메인별 보상값 정리")

# 보상 통계
rewards_values = [r["reward"] for r in rewards]
kv_print([
    ("평균 보상", f"{np.mean(rewards_values):.3f}"),
    ("표준편차", f"{np.std(rewards_values):.3f}"),
    ("최고 보상", f"{np.max(rewards_values):.3f}"),
    ("최저 보상", f"{np.min(rewards_values):.3f}")
])

print(f"→ 같은 '좋은 행동'도 도메인별로 보상이 다르다!")
print(f"    이것이 Task Perspective의 핵심이다.")

  [Step 2] 보상 분포 분석
    도메인별 보상값 정리
  평균 보상  0.729
  표준편차   0.147
  최고 보상  0.862
  최저 보상  0.504
→ 같은 '좋은 행동'도 도메인별로 보상이 다르다!
    이것이 Task Perspective의 핵심이다.


## 실습 4: Generative Agent Memory와 보상의 연관

메모리가 축적될수록 LLM 보상 평가가 어떻게 변하는지 보여준다.

In [11]:
heading("실습 4: 메모리와 보상의 동적 상호작용")

# Generative Agent Memory 사용
code_memory = MemoryStream()

# 코드 도메인의 예시 경험 저장
code_experiences = [
    ("def factorial(n): return 1 if n <= 1 else n * factorial(n-1)", "recursive", 0.8),
    ("테스트: assert factorial(5) == 120", "test", 0.9),
    ("반복 테스트 후 최적화: 런타임 50% 단축", "optimization", 0.7)
]

step_print(1, "메모리 구축", f"{len(code_experiences)}개 경험 저장")

for i, (content, mem_type, importance) in enumerate(code_experiences, 1):
    code_memory.add(content, importance=importance, mem_type=mem_type)
    print(f"  [{i}] {mem_type:12s} (중요도 {importance}): {content[:50]}...")

print(f"메모리 크기: {len(code_memory)}개 항목")


────────────────────────────────────────
  실습 4: 메모리와 보상의 동적 상호작용
────────────────────────────────────────
  [Step 1] 메모리 구축
    3개 경험 저장
  [1] recursive    (중요도 0.8): def factorial(n): return 1 if n <= 1 else n * fact...
  [2] test         (중요도 0.9): 테스트: assert factorial(5) == 120...
  [3] optimization (중요도 0.7): 반복 테스트 후 최적화: 런타임 50% 단축...
메모리 크기: 3개 항목


In [14]:
# 메모리를 바탕으로 LLM 반성(Reflect) 생성
step_print(2, "메모리 반성", "저장된 경험으로부터 통찰 도출")

reflection = code_memory.reflect("코드 최적화")
print(f"반성 결과: {reflection}")

print(f"→ 메모리를 바탕으로 LLM이 더 지능형 보상을 평가할 수 있다.")
print(f"    Generative Agent Memory가 Task-Specific 보상 형성의 핵심이다.")

  [Step 2] 메모리 반성
    저장된 경험으로부터 통찰 도출
반성 결과: 주어진 기억들을 종합하면, 재귀를 이용한 팩토리얼 함수의 성능을 반복 테스트 후 최적화하여 런타임을 50% 단축시킨 결과, 코드의 효율성을 크게 향상시킬 수 있음을 알 수 있다. 이는 알고리즘의 성능 개선이 실제 실행 시간에 미치는 긍정적인 영향을 보여준다.
→ 메모리를 바탕으로 LLM이 더 지능형 보상을 평가할 수 있다.
    Generative Agent Memory가 Task-Specific 보상 형성의 핵심이다.


## 요약

1. **Task Perspective는 메모리 구조를 결정한다**
   - 각 도메인은 고유한 메모리 아키텍처를 요구한다.
   - 코드 → 실행 기록, 검색 → 신뢰도, 수학 → 증명 단계

2. **메모리가 보상 형성을 가능하게 한다**
   - LLM은 메모리를 참고하여 더 나은 보상을 평가한다.
   - 과업 특화 보상 = LLM 평가(행동, 도메인별 기준, 메모리 컨텍스트)

3. **일반화와 특화의 트레이드오프**
   - 전이 학습은 메모리 구조의 불일치로 인해 실패한다.
   - 완전한 일반화는 불가능하고, 완전한 특화도 도메인 간 통찰을 잃는다.